In [1]:
import pandas as pd

# Load the generated dataset — this is the working set, not data/raw
customers = pd.read_csv("../data/generated/customers_data.csv")

print(customers.shape)
print(customers.dtypes)
print(customers.isnull().sum())


(1000, 8)
Customer_ID      int64
Name            object
Age              int64
Gender          object
Location        object
Join_Date       object
Churn            int64
Total_Spent    float64
dtype: object
Customer_ID    0
Name           0
Age            0
Gender         0
Location       0
Join_Date      0
Churn          0
Total_Spent    0
dtype: int64


### Interprétation — structure du dataset

- **1000 lignes, 8 colonnes**, aucune valeur manquante (`isnull().sum()` = 0 partout)
  → pas de nettoyage à prévoir pour cette table, on peut passer directement à l'analyse.
- Types cohérents : `Customer_ID` et `Churn` en `int64`, `Total_Spent` en `float64`,
  le reste en texte (`str`).
- ⚠️ `Customer_ID` est un **identifiant**, pas une variable. Il ne doit jamais être
  utilisé comme feature (aucun pouvoir prédictif réel, juste un numéro arbitraire) —
  à exclure explicitement avant l'étape modeling.

In [2]:
# Step 1: class balance — this decides which metrics matter later
churn_rate = customers["Churn"].value_counts(normalize=True)
print(churn_rate)

Churn
0    0.592
1    0.408
Name: proportion, dtype: float64


### Interprétation — équilibre des classes

Répartition : **59.2% non-churn (0) / 40.8% churn (1)**.

C'est un déséquilibre modéré, pas sévère (on est loin d'un cas 95/5). Conséquence
pratique : l'accuracy reste un indicateur à peu près lisible ici, mais on privilégiera
quand même F1 / ROC-AUC comme métriques principales, par prudence et parce que le
modèle sera comparé sur ces axes de toute façon. Pas besoin de rééquilibrage
(SMOTE, class_weight) à ce stade — le déséquilibre n'est pas assez marqué pour
le justifier d'emblée.

In [3]:
# Step 2: numeric columns split by churn — look for suspiciously clean separation
numeric_cols = customers.select_dtypes(include="number").columns.drop("Churn")
print(customers.groupby("Churn")[numeric_cols].describe().T)

Churn                         0            1
Customer_ID count    592.000000   408.000000
            mean    2503.363176  2496.345588
            std      291.241298   285.573918
            min     2001.000000  2004.000000
            25%     2251.750000  2248.750000
            50%     2505.500000  2488.500000
            75%     2756.500000  2738.500000
            max     3000.000000  2999.000000
Age         count    592.000000   408.000000
            mean      44.314189    44.906863
            std       15.199495    14.657135
            min       18.000000    18.000000
            25%       31.000000    33.000000
            50%       44.000000    46.000000
            75%       58.000000    57.000000
            max       70.000000    70.000000
Total_Spent count    592.000000   408.000000
            mean    8449.865845  1351.838946
            std     2612.559140   856.348655
            min     2200.440000    27.080000
            25%     6480.172500   675.487500
          

### Interprétation — pouvoir discriminant des colonnes numériques

- **Customer_ID** : moyennes quasi identiques (2503 vs 2496) — normal et attendu,
  puisque c'est un identifiant arbitraire. Confirme qu'il faut l'exclure des features
  (résultat sans signification, à ignorer pour l'analyse).

- **Age** : 44.3 ans (non-churn) vs 44.9 ans (churn) — quasiment aucune différence,
  écarts-types similaires (15.2 vs 14.7). **Age ne semble pas discriminant** pour
  prédire le churn dans ce dataset. À garder en tête : ce ne sera probablement
  pas une feature forte, mais on la garde quand même dans le pipeline (une
  variable faible n'est pas nuisible, sauf preuve du contraire).

- **Total_Spent** : ⚠️ **écart massif et suspect**.
  - Non-churn : moyenne 8450, médiane 8208
  - Churn : moyenne 1352, médiane 1187
  - Les non-churners dépensent en moyenne **6× plus** que les churners, avec un
    écart-type bien plus resserré côté churn (856 vs 2612).
  - Il y a un léger chevauchement (max churn = 5287 > min non-churn = 2200), donc
    la séparation n'est pas *parfaite*, mais elle est beaucoup trop nette pour être
    un simple signal comportemental naturel.

  **Confirme l'hypothèse de fuite de données posée dans `docs/churn.md`** : comme
  `Churn` est dérivé de `_Behavior`, et que `_Behavior` a probablement aussi piloté
  le volume d'achats généré par `generate_data.py`, `Total_Spent` encode indirectement
  la cible elle-même plutôt qu'un comportement observable *avant* le churn.
  → À documenter comme risque de leakage confirmé, et à traiter avant modeling
  (soit exclusion, soit transformation en feature moins directe — ex. tendance de
  dépense sur les N derniers mois plutôt que le total).

### ⚠️ Leakage détecté sur Total_Spent

L'écart ×6 entre churners et non-churners n'est pas un signal métier : en
lisant `generate_data.py`, `_Behavior` pilote directement le nombre d'achats,
la quantité par achat et l'accès aux produits premium — les 3 leviers qui
construisent Total_Spent. Cette variable encode donc la cible plutôt que de
la prédire.

→ Décision : Total_Spent exclu des features pour M6.
→ Détail complet : voir docs/churn.md (entrée du 10 septembre 2026)

In [4]:
sales = pd.read_csv("../data/generated/sales_data.csv", parse_dates=["Date"])

print(sales.shape)
print(sales.dtypes)
print(sales["Date"].min(), "→", sales["Date"].max())
print(sales["Customer_ID"].nunique(), "clients uniques sur", customers.shape[0])

(15741, 8)
Sale_ID                 int64
Product_ID              int64
Customer_ID             int64
Date           datetime64[ns]
Quantity                int64
Sale_Price            float64
Channel                object
Campaign_ID             int64
dtype: object
2022-01-09 00:00:00 → 2025-12-31 00:00:00
1000 clients uniques sur 1000


### Interprétation — structure de sales_data.csv

- **15 741 transactions**, 8 colonnes, aucun type suspect (Date bien en
  datetime, pas besoin de parsing supplémentaire).
- Période : **2022-01-09 → 2025-12-31**, cohérente avec START_DATE/END_DATE
  du générateur.
- **1000 clients uniques sur 1000** → chaque client a au moins un achat.
  Confirme que le Total_Spent = 0 observé pour certains clients (via le
  fillna(0) dans generate_data.py) n'existe pas dans cet échantillon —
  pas de cas "client sans historique d'achat" à gérer pour l'instant.

In [5]:
reference_date = sales["Date"].max()
print("Date de référence choisie :", reference_date)

Date de référence choisie : 2025-12-31 00:00:00


### Décision — date de référence pour le calcul RFM

Date de référence retenue : **2025-12-31** (= max(Date) dans sales_data.csv),
unique pour tous les clients.

Rejeté : date de référence par client (type last_purchase_limit du générateur)
— dérivée de _Behavior, donc leakage direct, même mécanisme que Total_Spent.

In [6]:
rfm = sales.groupby("Customer_ID").agg(
    Recency=("Date", lambda x: (reference_date - x.max()).days),
    Frequency=("Sale_ID", "count"),
    Monetary=("Quantity", lambda x: (x * sales.loc[x.index, "Sale_Price"]).sum())
).reset_index()

print(rfm.describe())

       Customer_ID      Recency    Frequency      Monetary
count  1000.000000  1000.000000  1000.000000   1000.000000
mean   2500.500000   242.729000    15.741000   5553.870870
std     288.819436   279.284667     8.403365   4064.221665
min    2001.000000     0.000000     1.000000     27.080000
25%    2250.750000    19.000000     8.000000   1475.352500
50%    2500.500000    82.500000    16.000000   5715.215000
75%    2750.250000   442.250000    22.000000   8739.757500
max    3000.000000  1176.000000    37.000000  16856.640000


### Interprétation — distribution RFM

- **Recency** : min 0 (achat juste avant le 31/12), max 1176 jours (cohérent
  avec la période totale ~1450 jours). Écart-type élevé (279) par rapport à
  la moyenne (243) → distribution probablement bimodalez (clients récents vs
  clients inactifs depuis longtemps).
- **Frequency** : moyenne 15.7, cohérente avec la moyenne pondérée théorique
  du générateur (~15.6 en combinant les 3 comportements).
- **Monetary** : moyenne 5554, quasi identique à la moyenne pondérée observée
  sur Total_Spent (5553) → confirme que la formule (Quantity × Sale_Price,
  sommée par client) est correcte.

In [7]:
rfm_with_churn = rfm.merge(customers[["Customer_ID", "Churn"]], on="Customer_ID")
print(rfm_with_churn.groupby("Churn")[["Recency", "Frequency", "Monetary"]].describe().T)

Churn                       0            1
Recency   count    592.000000   408.000000
          mean      42.692568   532.977941
          std       51.952244   211.850585
          min        0.000000   168.000000
          25%        9.000000   362.000000
          50%       26.000000   503.500000
          75%       58.250000   671.500000
          max      479.000000  1176.000000
Frequency count    592.000000   408.000000
          mean      21.368243     7.575980
          std        5.549306     3.965343
          min        9.000000     1.000000
          25%       17.000000     5.000000
          50%       21.000000     7.000000
          75%       25.000000    10.000000
          max       37.000000    21.000000
Monetary  count    592.000000   408.000000
          mean    8449.865845  1351.838946
          std     2612.559140   856.348655
          min     2200.440000    27.080000
          25%     6480.172500   675.487500
          50%     8207.965000  1187.450000
          7

### Interprétation — RFM par classe de Churn

- **Recency** : 43 jours (non-churn) vs 533 jours (churn) — séparation très nette,
  quasi pas de chevauchement (max non-churn 479 vs min churn 168, chevauchement
  léger seulement).
- **Frequency** : 21.4 achats (non-churn) vs 7.6 (churn) — séparation nette aussi.
- **Monetary** : identique à Total_Spent (8450 vs 1352) — logique, même formule.

Les 3 variables séparent bien les classes. Question à trancher : est-ce un
signal légitime ou un leakage comme Total_Spent ?

### Décision — RFM retenu comme feature set légitime

Contrairement à Total_Spent (colonne pré-calculée, fenêtre temporelle inconnue),
Recency/Frequency/Monetary sont calculées par nous-mêmes, avec une date de
référence connue et documentée. Le générateur documente explicitement les RFM
comme feature set prévu pour M3/M6 — la forte séparation par classe est
attendue, pas suspecte.

**Limite du dataset à noter pour le rapport (M9)** : chaque client a un
comportement figé sur toute sa période (pas de bascule "actif → churné" à
une date précise). Le modèle fera donc de la classification d'état actuel,
pas de la prédiction précoce d'un événement futur.

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.append(str(SRC_PATH))

from churn.features import build_feature_set

reference_date = pd.Timestamp("2025-12-31")
features = build_feature_set(customers, sales, reference_date)
features.head()

,Customer_ID,Age,Gender,Location,Churn,Recency,Frequency,Monetary
0,2001,45,Female,Jacksonville,0,28,26,9757.66
1,2002,50,Male,San Antonio,0,64,26,10921.86
2,2003,63,Female,New York,0,37,23,7032.84
3,2004,52,Male,Philadelphia,1,361,3,487.76
4,2005,60,Female,San Jose,0,11,21,9999.42


In [12]:
from churn.features import build_feature_set, encode_categorical_features

features = build_feature_set(customers, sales, reference_date)
features_encoded = encode_categorical_features(features)

print(features_encoded.shape)
print(features_encoded.columns.tolist())

(1000, 18)
['Customer_ID', 'Age', 'Churn', 'Recency', 'Frequency', 'Monetary', 'Gender_Male', 'Location_Chicago', 'Location_Dallas', 'Location_Houston', 'Location_Jacksonville', 'Location_Los Angeles', 'Location_New York', 'Location_Philadelphia', 'Location_Phoenix', 'Location_San Antonio', 'Location_San Diego', 'Location_San Jose']


## Conclusion — Feature set validé pour M6

**Décisions prises dans ce notebook :**
- `Total_Spent` (customers_data.csv) exclu — leakage confirmé via `_Behavior`
  (voir génération dans generate_data.py, section 4)
- Date de référence retenue : 2025-12-31 (max(Date), unique pour tous les
  clients — évite le leakage d'une date par client)
- Feature set retenu : Recency, Frequency, Monetary (calculés depuis
  sales_data.csv), Age, Gender, Location
- Encodage validé : one-hot avec `drop_first=True` sur Gender/Location
  (18 colonnes finales)

**Limite du dataset notée pour le rapport (M9)** : comportement client figé
sur toute la période, pas de bascule temporelle — classification d'état,
pas prédiction précoce.

**Suite du travail** : comparaison de modèles (Logistic Regression, Random
Forest, XGBoost) dans `notebooks/model_training.ipynb`, qui reprend
`build_feature_set()` et `encode_categorical_features()` depuis
`src/churn/features.py`.